# 🏎️ F1 BI — Pipeline ETL
## Proyecto Final · Módulo 4: Inteligencia de Negocios y SQL Avanzado

**Dataset:** Ergast F1 Database (1950–2024)  
**Destino:** Aurora PostgreSQL — Schema `f1_dw`  
**Grano:** Un registro por piloto por carrera

| Fase | Descripción |
|---|---|
| Extract | Lee los 14 CSVs del dataset Ergast |
| Transform | Construye dimensiones y fact table |
| Load | Upsert idempotente en Aurora |
| Validate | Conteos, integridad referencial, nulos |

## 0. Instalación de dependencias

Instala las 4 librerías que necesita el pipeline. Se ejecuta solo una vez en Google Colab porque el entorno es efímero y no tiene las librerías preinstaladas.

> pandas — manipulación y transformación de los DataFrames (los CSVs)

> sqlalchemy — conexión y ejecución de SQL contra Aurora PostgreSQL

> psycopg2-binary — driver que SQLAlchemy usa internamente para hablar con PostgreSQL

> python-dotenv — lee las credenciales del archivo .env sin hardcodearlas

In [1]:
# Ejecutar solo la primera vez en Google Colab
!pip install pandas sqlalchemy psycopg2-binary python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 26.5 MB/s eta 0:00:00


## 1. Imports y logging

Verificamos que se importen las librerias necesarias de manera correcta.

In [2]:
import os, logging
import pandas as pd
from dotenv import load_dotenv
import sqlalchemy # Import the sqlalchemy module
import psycopg2   # Import the psycopg2 module
from sqlalchemy import create_engine, text

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('f1_etl')
logger.info('Imports cargados correctamente')

print(f"pandas      {pd.__version__}")
print(f"sqlalchemy  {sqlalchemy.__version__}")
print(f"psycopg2    {psycopg2.__version__.split()[0]}")

pandas      2.2.2
sqlalchemy  2.0.50
psycopg2    2.9.12


Importa todo lo que se usará en el resto del notebook. Configura logging para que cada paso del ETL imprima un mensaje con timestamp, nivel y descripción. El objeto logger es el que aparece en todas las funciones como logger.info(...).

> ¿Por qué logging y no print? Con print no sabes cuándo ocurrió algo ni si fue un error o info. El logging permite filtrar por nivel (INFO, WARNING, ERROR) y guardar en archivo si se necesita auditoría.

# 1.1 Conexión a Aurora PostgreSQL con SQLAlchemy

Se creo una llave secreta en Google Colab para guardar la *Constrasena Mestra*, *Punto de Enlace*, *Nombre de la Base de Datos* y *Nombre de Usuario Maestro* creadas desde el inicio de la base de datos y su cluster en *AWS*. Contemplando las mejores practicas y manteniendo la confidencialidad o seguridad de la base de datos.

Desde Google Colab, *Secrets* permite almacenar "variables de entorno, rutas de acceso a archivos o claves para configurar tu código. Los valores almacenados aquí son privados, y visibles únicamente para el propietario la cuenta y en los notebooks seleccionados".

In [44]:
# Importa las librerías necesarias
from sqlalchemy import create_engine, text
from google.colab import userdata

# Recupera tus secretos desde Google Colab
AURORA_HOST     = userdata.get('F1_HOST')
AURORA_PASSWORD = userdata.get('F1_data')
AURORA_USER     = userdata.get('AURORA_USER')

# Crea el engine de conexión con SQLAlchemy
# DB_NAME = postgres (base de datos real del cluster Aurora)
engine = create_engine(
    f"postgresql+psycopg2://{AURORA_USER}:{AURORA_PASSWORD}@{AURORA_HOST}:5432/postgres",
    pool_pre_ping=True
)

# Verificar conexión
with engine.connect() as conn:
    db = conn.execute(text("SELECT current_database()")).scalar()
    print(f"✅ Conectado a: {db}")
    # Debe devolver: postgres

✅ Conectado a: postgres


a) **Primera pueba de conexion:**

In [45]:
# Prueba la conexión ejecutando una consulta simple
with engine.connect() as conn:
    result = conn.execute(text("SELECT NOW();"))
    for row in result:
        print("Conexión exitosa, fecha/hora en Aurora:", row[0])


Conexión exitosa, fecha/hora en Aurora: 2026-06-10 01:38:41.675581+00:00


*Resultado esperado*:

> Conexión exitosa, fecha/hora en Aurora: 2026-06-08 01:38:20.863616+00:00

b) **Segunda prueba de conexion:**

*Smoke test* — confirmar que el engine conecta y la base responde:

In [46]:
with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar()
    print(version)

PostgreSQL 17.7 on x86_64-pc-linux-gnu, compiled by x86_64-pc-linux-gnu-gcc (GCC) 10.5.0, 64-bit


*Resultado esperado:*

> PostgreSQL 17.7 on x86_64-pc-linux-gnu, compiled by x86_64-pc-linux-gnu-gcc (GCC) 10.5.0, 64-bit

> Si ves algo como PostgreSQL 17.x on aarch64-unknown-linux-gnu..., la conexión funciona.

## 2. IMPORTAR DATOS

Se importan los datos, el cual define DATA_PATH (dónde están los CSVs) directamente desde *kagglehub*.

Descarga el dataset directo desde Kaggle con una línea. Más limpio, reproducible y no depende de que tengas los archivos en Drive. Cualquier persona que clone el repositorio puede correr el notebook sin preparación previa.

In [8]:
import kagglehub
DATA_PATH = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")


Using Colab cache for faster access to the 'formula-1-world-championship-1950-2020' dataset.


**2.2 Exploramos el contenido de F1 race data from 1950 to 2024 con los siguientes codigos:**


In [9]:
# para listar los archivos ejecutamos el siguiente codigo:
print("Contenido del dataset:")
print(os.listdir(DATA_PATH))


Contenido del dataset:
['races.csv', 'constructor_results.csv', 'drivers.csv', 'constructors.csv', 'lap_times.csv', 'status.csv', 'driver_standings.csv', 'seasons.csv', 'pit_stops.csv', 'sprint_results.csv', 'constructor_standings.csv', 'results.csv', 'circuits.csv', 'qualifying.csv']


In [10]:
# listar los archivos con sus tamaños.
!ls -lh {path}

ls: cannot access '{path}': No such file or directory


In [11]:
# Ver estructura completa: subcarpetas y archivos
!apt-get install tree
!tree {DATA_PATH}

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (757 kB/s)
Selecting previously unselected package tree.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...
/kaggle/input/formula-1-world-championship-1950-2020
├── circuits.csv
├── constructor_results.csv
├── constructors.csv
├── constructor_standings.csv
├── drivers.csv
├── driver_standings.csv
├── lap_times.csv
├── pit_stops.csv
├── qualifying.csv
├── races.csv
├── results.csv
├── seasons.cs

In [12]:
# para ver el contenido de cada data set y ver de que manera se relacionan
# Listar todos los archivos dentro del dataset
files = os.listdir(DATA_PATH)
print("Archivos disponibles en el dataset:\n", files)

# Leer cada archivo CSV y mostrar información básica
for file in files:
    if file.endswith(".csv"):
        print("\n=== Archivo:", file, "===")
        df = pd.read_csv(os.path.join(DATA_PATH, file))
        print("Columnas:", df.columns.tolist())
        print("Primeras filas:")
        print(df.head(3))  # muestra las primeras 3 filas

Archivos disponibles en el dataset:
 ['races.csv', 'constructor_results.csv', 'drivers.csv', 'constructors.csv', 'lap_times.csv', 'status.csv', 'driver_standings.csv', 'seasons.csv', 'pit_stops.csv', 'sprint_results.csv', 'constructor_standings.csv', 'results.csv', 'circuits.csv', 'qualifying.csv']

=== Archivo: races.csv ===
Columnas: ['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time']
Primeras filas:
   raceId  year  round  circuitId                   name        date  \
0       1  2009      1          1  Australian Grand Prix  2009-03-29   
1       2  2009      2          2   Malaysian Grand Prix  2009-04-05   
2       3  2009      3         17     Chinese Grand Prix  2009-04-19   

       time                                                url fp1_date  \
0  06:00:00  http://en.wikipedia.org/wiki/2009_Australian_G...       \N   
1  09

In [13]:
# Mostrar dimensiones de cada archivo CSV
for file in files:
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(DATA_PATH, file))
        print(f"Archivo: {file}")
        print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas\n")

Archivo: races.csv
Dimensiones: 1125 filas x 18 columnas

Archivo: constructor_results.csv
Dimensiones: 12625 filas x 5 columnas

Archivo: drivers.csv
Dimensiones: 861 filas x 9 columnas

Archivo: constructors.csv
Dimensiones: 212 filas x 5 columnas

Archivo: lap_times.csv
Dimensiones: 589081 filas x 6 columnas

Archivo: status.csv
Dimensiones: 139 filas x 2 columnas

Archivo: driver_standings.csv
Dimensiones: 34863 filas x 7 columnas

Archivo: seasons.csv
Dimensiones: 75 filas x 2 columnas

Archivo: pit_stops.csv
Dimensiones: 11371 filas x 7 columnas

Archivo: sprint_results.csv
Dimensiones: 360 filas x 16 columnas

Archivo: constructor_standings.csv
Dimensiones: 13391 filas x 7 columnas

Archivo: results.csv
Dimensiones: 26759 filas x 18 columnas

Archivo: circuits.csv
Dimensiones: 77 filas x 9 columnas

Archivo: qualifying.csv
Dimensiones: 10494 filas x 9 columnas



En *00_eda.ipynb* dentro de la carpeta **scripts/** se encuentra el Analisis de Datos Exploratorio (EDA). Lo cual justifica con evidencia real del *F1 race data from 1950 to 2024* el desarrollo implementado en etl_pipeline.ipynb de este Notebook.

Se explorar el dataset antes del ETL para identificar:
- Cobertura y completitud de cada tabla
- Valores nulos, atípicos e inconsistencias
- Decisiones de limpieza a implementar en `transform_fact()`

| Sección | Qué analiza |
|---|---|
| 1. Setup | Instalación, imports y carga de datos |
| 2. Cobertura | Filas, columnas y % nulos por tabla |
| 3. Nulos críticos | Nulos en columnas clave de results.csv |
| 4. Pit stops | Cobertura histórica (solo desde 1994) |
| 5. Outliers | Posición de salida, delta, tiempos de vuelta |
| 6. Consistencia | IDs huérfanos entre tablas |
| 7. Distribuciones | Puntos, abandonos y carreras por era |
| 8. Decisiones ETL | Resumen de limpiezas a aplicar |


## 3. EXTRACT — Lectura de CSVs

*extract(data_path)* — Lee los 14 CSVs del dataset Ergast:

Itera sobre un diccionario de 14 archivos CSV, los lee con pd.read_csv() y los guarda en el diccionario raw. Si un archivo no existe lo avisa con logger.warning pero no detiene el pipeline. Retorna raw, que es el punto de partida para todas las transformaciones.

na_values=["\\N","NA","","NULL"]: el dataset Ergast usa \N para representar valores nulos (convención de MySQL). Sin esto, esos valores quedarían como texto en lugar de NaN.

> Salida esperada: un log por cada archivo con el nombre, cantidad de filas y columnas.

In [47]:
def extract(data_path):
    archivos = {
        'races':'races.csv','results':'results.csv','drivers':'drivers.csv',
        'constructors':'constructors.csv','circuits':'circuits.csv',
        'pit_stops':'pit_stops.csv','lap_times':'lap_times.csv',
        'qualifying':'qualifying.csv','driver_standings':'driver_standings.csv',
        'constructor_standings':'constructor_standings.csv',
        'constructor_results':'constructor_results.csv','seasons':'seasons.csv',
        'status':'status.csv','sprint_results':'sprint_results.csv',
    }
    raw = {}
    for nombre, archivo in archivos.items():
        ruta = os.path.join(data_path, archivo)
        if not os.path.exists(ruta):
            logger.warning(f'  No encontrado (omitido): {ruta}')
            continue
        df = pd.read_csv(ruta, na_values=['\\N','NA','','NULL'])
        raw[nombre] = df
        logger.info(f'  {archivo:<40} -> {len(df):>8,} filas')
    return raw

raw = extract(DATA_PATH)

## 4. TRANSFORM — Dimensiones

*transform_fact()* — Construye fact_resultado_carrera:

Es la transformación más compleja. Parte de results.csv y hace varios pasos en cadena para producir la fact table final.

- Paso 1 — Pit stops: agrupa pit_stops.csv por raceId + driverId y cuenta las paradas. Este conteo se une a results para tener num_pitstops por piloto/carrera.

- Paso 2 — Resolver SKs: hace 5 merges consecutivos para reemplazar los IDs naturales de Ergast (driverId, constructorId, etc.) por los surrogate keys generados en la celda anterior. Esto es el corazón del ETL dimensional.

- Paso 3 — Medidas derivadas: calcula delta_posicion (posición ganada o perdida respecto a la salida), es_abandono (True si el estado no es "Finalizado") y convierte milisegundos a numérico.

- Paso 4 — Limpieza final: descarta filas donde alguna SK quedó sin resolver (no debería ocurrir, pero es un safety net) y convierte las SKs a entero.

fact.head(): muestra las primeras 5 filas en Colab para inspección visual inmediata.

In [48]:
def asignar_era(y):
    if pd.isna(y) or y<=1966: return 'Aspiracion natural 1950-66'
    elif y<=1976: return 'Era DFV 1967-76'
    elif y<=1988: return 'Era turbo 1977-88'
    elif y<=2005: return 'NA moderno 1989-2005'
    elif y<=2013: return 'Era V8 2006-13'
    else:         return 'Era hibrida 2014+'

def transform_dim_piloto(raw):
    df = raw['drivers'].copy()
    df['nombre_completo'] = (df['forename'].fillna('')+' '+df['surname'].fillna('')).str.strip()
    df['fecha_nacimiento'] = pd.to_datetime(df['dob'], errors='coerce')
    dim = df.rename(columns={'driverId':'driver_id','driverRef':'driver_ref',
        'number':'numero_permanente','code':'codigo_piloto','nationality':'nacionalidad'})
    dim = dim[['driver_id','driver_ref','numero_permanente','codigo_piloto',
               'nombre_completo','fecha_nacimiento','nacionalidad']]
    dim = dim.sort_values('driver_id').reset_index(drop=True)
    dim.insert(0,'piloto_sk',dim.index+1)
    logger.info(f'dim_piloto:      {len(dim):,} filas'); return dim

def transform_dim_constructor(raw):
    df = raw['constructors'].copy()
    if 'constructor_results' in raw and 'races' in raw:
        cr = raw['constructor_results'].merge(raw['races'][['raceId','year']],on='raceId',how='left')
        em = cr.groupby('constructorId')['year'].median().reset_index()
        em.columns=['constructorId','my']
        em['era_f1'] = em['my'].apply(asignar_era)
        df = df.merge(em[['constructorId','era_f1']],on='constructorId',how='left')
    else:
        df['era_f1'] = 'Desconocido'
    dim = df.rename(columns={'constructorId':'constructor_id','constructorRef':'constructor_ref',
        'name':'nombre','nationality':'nacionalidad'})
    dim = dim[['constructor_id','constructor_ref','nombre','nacionalidad','era_f1']]
    dim = dim.sort_values('constructor_id').reset_index(drop=True)
    dim.insert(0,'constructor_sk',dim.index+1)
    logger.info(f'dim_constructor: {len(dim):,} filas'); return dim

def transform_dim_circuito(raw):
    df = raw['circuits'].copy()
    dim = df.rename(columns={'circuitId':'circuit_id','circuitRef':'circuit_ref',
        'name':'nombre','location':'localidad','country':'pais','lat':'latitud','lng':'longitud'})
    dim = dim[['circuit_id','circuit_ref','nombre','localidad','pais','latitud','longitud']]
    dim['latitud']  = pd.to_numeric(dim['latitud'],  errors='coerce')
    dim['longitud'] = pd.to_numeric(dim['longitud'], errors='coerce')
    dim = dim.sort_values('circuit_id').reset_index(drop=True)
    dim.insert(0,'circuito_sk',dim.index+1)
    logger.info(f'dim_circuito:    {len(dim):,} filas'); return dim

def transform_dim_tiempo(raw):
    df = raw['races'].copy()
    df['era_f1'] = df['year'].apply(asignar_era)
    df['fecha']  = pd.to_datetime(df['date'], errors='coerce')
    dim = df.rename(columns={'raceId':'race_id','year':'anio','name':'nombre_gp','round':'numero_ronda'})
    dim = dim[['race_id','fecha','anio','nombre_gp','numero_ronda','era_f1']].copy()
    dim['temporada'] = dim['anio']
    dim = dim.sort_values('race_id').reset_index(drop=True)
    dim.insert(0,'tiempo_sk',dim.index+1)
    logger.info(f'dim_tiempo:      {len(dim):,} filas'); return dim

def transform_dim_estado(raw):
    df = raw['status'].copy()
    mec=['Engine','Gearbox','Transmission','Clutch','Hydraulics','Electrical','Brakes',
         'Suspension','Overheating','Mechanical','Turbo','ERS','Battery','Tyre','Puncture',
         'Wheel','Driveshaft','Steering','Power Unit','Pneumatics','Oil pressure']
    acc=['Accident','Collision','Spun off','Collision damage','Fatal accident']
    des=['Disqualified','Excluded','Not classified','Did not qualify','107% rule','Underweight']
    def cat(s):
        if s=='Finished' or '+' in str(s) or 'Lap' in str(s): return 'Finalizado'
        elif any(m.lower() in s.lower() for m in mec): return 'Abandono mecanico'
        elif any(a.lower() in s.lower() for a in acc): return 'Accidente'
        elif any(d.lower() in s.lower() for d in des): return 'Descalificado'
        else: return 'Otro'
    df['categoria'] = df['status'].apply(cat)
    dim = df.rename(columns={'statusId':'status_id','status':'descripcion'})
    dim = dim[['status_id','descripcion','categoria']]
    dim = dim.sort_values('status_id').reset_index(drop=True)
    dim.insert(0,'estado_sk',dim.index+1)
    logger.info(f'dim_estado:      {len(dim):,} filas')
    logger.info(f'Categorias:\n{dim["categoria"].value_counts().to_string()}')
    return dim

dim_piloto      = transform_dim_piloto(raw)
dim_constructor = transform_dim_constructor(raw)
dim_circuito    = transform_dim_circuito(raw)
dim_tiempo      = transform_dim_tiempo(raw)
dim_estado      = transform_dim_estado(raw)

## 5. TRANSFORM — Fact Table

In [49]:
def transform_fact(raw, dim_piloto, dim_constructor, dim_circuito, dim_tiempo, dim_estado):
    results = raw['results'].copy()
    races   = raw['races'][['raceId','circuitId']].copy()
    if 'pit_stops' in raw:
        pc = raw['pit_stops'].groupby(['raceId','driverId']).size().reset_index(name='num_pitstops')
        results = results.merge(pc, on=['raceId','driverId'], how='left')
        results['num_pitstops'] = results['num_pitstops'].fillna(0).astype(int)
    else:
        results['num_pitstops'] = 0
    results = results.merge(races, on='raceId', how='left')
    results = results.merge(dim_piloto[['piloto_sk','driver_id']],             left_on='driverId',      right_on='driver_id',      how='left')
    results = results.merge(dim_constructor[['constructor_sk','constructor_id']],left_on='constructorId', right_on='constructor_id', how='left')
    results = results.merge(dim_circuito[['circuito_sk','circuit_id']],         left_on='circuitId',     right_on='circuit_id',     how='left')
    results = results.merge(dim_tiempo[['tiempo_sk','race_id']],                left_on='raceId',        right_on='race_id',        how='left')
    results = results.merge(dim_estado[['estado_sk','status_id']],              left_on='statusId',      right_on='status_id',      how='left')
    results['posicion_salida']     = pd.to_numeric(results['grid'],          errors='coerce')
    results['posicion_final']      = pd.to_numeric(results['positionOrder'], errors='coerce')
    results['puntos']              = pd.to_numeric(results['points'],        errors='coerce').fillna(0)
    results['vueltas_completadas'] = pd.to_numeric(results['laps'],          errors='coerce').fillna(0).astype(int)
    results['tiempo_total_ms']     = pd.to_numeric(results['milliseconds'],  errors='coerce')
    results['delta_posicion']      = results['posicion_salida'] - results['posicion_final']
    fin = dim_estado[dim_estado['categoria']=='Finalizado']['estado_sk'].tolist()
    results['es_abandono'] = ~results['estado_sk'].isin(fin)
    fact = results[['piloto_sk','constructor_sk','circuito_sk','tiempo_sk','estado_sk',
        'posicion_salida','posicion_final','puntos','vueltas_completadas',
        'num_pitstops','tiempo_total_ms','delta_posicion','es_abandono']].copy()
    fact = fact.dropna(subset=['piloto_sk','constructor_sk','circuito_sk','tiempo_sk','estado_sk'])
    for col in ['piloto_sk','constructor_sk','circuito_sk','tiempo_sk','estado_sk']:
        fact[col] = fact[col].astype(int)
    logger.info(f'fact_resultado_carrera: {len(fact):,} filas')
    return fact

fact = transform_fact(raw, dim_piloto, dim_constructor, dim_circuito, dim_tiempo, dim_estado)
fact.head()

,piloto_sk,constructor_sk,circuito_sk,tiempo_sk,estado_sk,posicion_salida,posicion_final,puntos,vueltas_completadas,num_pitstops,tiempo_total_ms,delta_posicion,es_abandono
0,1,1,1,18,1,1,1,10.0,58,0,5690616.0,0,False
1,2,2,1,18,1,5,2,8.0,58,0,5696094.0,3,False
2,3,3,1,18,1,7,3,6.0,58,0,5698779.0,4,False
3,4,4,1,18,1,11,4,5.0,58,0,5707797.0,7,False
4,5,1,1,18,1,3,5,4.0,58,0,5708630.0,-2,False


## 6. LOAD — Carga en Aurora PostgreSQL

*load_table()* y *load_fact()* — Carga idempotente en Aurora:

Dos funciones con estrategias distintas de upsert, seguidas de las 6 llamadas de carga en el orden correcto (dimensiones primero, fact al final, para respetar las foreign keys).

load_table para dimensiones: hace TRUNCATE + INSERT. Borra toda la tabla y la recarga limpia. Es seguro porque las dimensiones son pequeñas (máximo ~1,100 filas) y no hay dependencias entre ellas en la misma operación.

load_fact para la fact table: usa ON CONFLICT (piloto_sk, tiempo_sk) DO UPDATE SET .... Si ya existe ese registro (mismo piloto, misma carrera) lo actualiza en lugar de duplicarlo. Esto es el upsert real de PostgreSQL.

Lotes de 1,000 filas: en lugar de insertar 26,000 filas en una sola transacción (que podría dar timeout), inserta en bloques de 1,000. Más estable en conexiones de red.

¿Por qué orden importa? La fact table tiene foreign keys a las 5 dimensiones. Si cargas la fact antes que las dims, PostgreSQL rechaza los INSERTs porque los SKs referenciados aún no existen.

In [60]:
# verifica que el esquema y las tablas se haya creado en AURORA
from sqlalchemy import text

def check_schema_and_tables(engine, schema='f1_dw'):
    query = text("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_schema = :schema
        ORDER BY table_name;
    """)
    with engine.connect() as conn:
        result = conn.execute(query, {"schema": schema}).fetchall()
        if not result:
            print(f"⚠️ El esquema '{schema}' no existe o no tiene tablas.")
        else:
            print(f"✅ Tablas encontradas en el esquema '{schema}':")
            for row in result:
                print(f" - {row.table_name}")

# Ejecuta la verificación
check_schema_and_tables(engine, schema='f1_dw')


✅ Tablas encontradas en el esquema 'f1_dw':
 - dim_circuito
 - dim_constructor
 - dim_estado
 - dim_piloto
 - dim_tiempo
 - fact_resultado_carrera


In [62]:
# verifica que el esquema y las tablas se haya creado en AURORA
from sqlalchemy import text

def check_schema_and_tables(engine, schema='f1_dw'):
    query = text("""
        SELECT t.table_schema, t.table_name, c.column_name, c.data_type
        FROM information_schema.tables t
        JOIN information_schema.columns c
          ON t.table_name = c.table_name AND t.table_schema = c.table_schema
        WHERE t.table_schema = :schema
        ORDER BY t.table_name, c.ordinal_position;
    """)
    with engine.connect() as conn:
        result = conn.execute(query, {"schema": schema}).fetchall()
        if not result:
            print(f"⚠️ El esquema '{schema}' no existe o no tiene tablas.")
        else:
            current_table = None
            for row in result:
                if row.table_name != current_table:
                    current_table = row.table_name
                    print(f"\n✅ Tabla: {row.table_schema}.{row.table_name}")
                print(f"   - {row.column_name} ({row.data_type})")

# Ejecuta la verificación
check_schema_and_tables(engine, schema='f1_dw')



✅ Tabla: f1_dw.dim_circuito
   - circuito_sk (integer)
   - circuit_id (integer)
   - circuit_ref (character varying)
   - nombre (character varying)
   - localidad (character varying)
   - pais (character varying)
   - latitud (numeric)
   - longitud (numeric)
   - created_at (timestamp without time zone)

✅ Tabla: f1_dw.dim_constructor
   - constructor_sk (integer)
   - constructor_id (integer)
   - constructor_ref (character varying)
   - nombre (character varying)
   - nacionalidad (character varying)
   - era_f1 (character varying)
   - created_at (timestamp without time zone)

✅ Tabla: f1_dw.dim_estado
   - estado_sk (integer)
   - status_id (integer)
   - descripcion (character varying)
   - categoria (character varying)
   - created_at (timestamp without time zone)

✅ Tabla: f1_dw.dim_piloto
   - piloto_sk (integer)
   - driver_id (integer)
   - driver_ref (character varying)
   - numero_permanente (integer)
   - codigo_piloto (character)
   - nombre_completo (character varyin

In [63]:
for nombre, df in [
    ('dim_piloto',      dim_piloto),
    ('dim_constructor', dim_constructor),
    ('dim_circuito',    dim_circuito),
    ('dim_tiempo',      dim_tiempo),
    ('dim_estado',      dim_estado),
    ('fact',            fact),
]:
    print(f"{nombre:<25} {len(df):>8,} filas")

dim_piloto                     861 filas
dim_constructor                212 filas
dim_circuito                    77 filas
dim_tiempo                   1,125 filas
dim_estado                     139 filas
fact                        26,759 filas


In [65]:
# ── LOAD con pandas.to_sql + method='multi' ──────────────────────────────────
import time

def load_table(df, table_name, engine, schema='f1_dw'):
    """
    Carga una dimensión con TRUNCATE + to_sql(method='multi').
    - TRUNCATE vacía la tabla antes de recargar (idempotente).
    - method='multi' agrupa múltiples filas en un solo INSERT,
      reduciendo round-trips a Aurora hasta 10x vs INSERT fila por fila.
    """
    full = f'{schema}.{table_name}'

    # 1. Vaciar tabla respetando constraints (CASCADE)
    with engine.begin() as conn:
        conn.execute(text(f'TRUNCATE TABLE {full} CASCADE'))

    # 2. Insertar con to_sql
    df.to_sql(
        name=table_name,
        con=engine,
        schema=schema,
        if_exists='append',
        index=False,
        chunksize=500,
        method='multi',
    )
    logger.info(f'  OK {len(df):,} filas -> {full}')


def load_fact(df, engine, schema='f1_dw'):
    """
    Carga la fact table con TRUNCATE + to_sql(method='multi').
    - Elimina duplicados internos del DataFrame antes de insertar.
    - TRUNCATE garantiza idempotencia sin conflictos de PK.
    """
    full = f'{schema}.fact_resultado_carrera'

    # 1. Eliminar duplicados del DataFrame por PK compuesta
    antes = len(df)
    df = df.drop_duplicates(subset=['piloto_sk', 'tiempo_sk'], keep='first')
    eliminados = antes - len(df)
    if eliminados > 0:
        logger.warning(f'  {eliminados} duplicados eliminados del DataFrame antes de cargar')

    # 2. Vaciar fact table
    with engine.begin() as conn:
        conn.execute(text(f'TRUNCATE TABLE {full}'))

    # 3. Insertar con to_sql
    df.to_sql(
        name='fact_resultado_carrera',
        con=engine,
        schema=schema,
        if_exists='append',
        index=False,
        chunksize=500,
        method='multi',
    )
    logger.info(f'  OK {len(df):,} filas -> {full}')


# ── Ejecutar carga en orden (dimensiones primero, fact al final) ──────────────

for nombre, df_dim in [
    ('dim_piloto',      dim_piloto),
    ('dim_constructor', dim_constructor),
    ('dim_circuito',    dim_circuito),
    ('dim_tiempo',      dim_tiempo),
    ('dim_estado',      dim_estado),
]:
    t0 = time.time()
    load_table(df_dim, nombre, engine)
    logger.info(f'    → {nombre}: {time.time()-t0:.1f}s')

t0 = time.time()
load_fact(fact, engine)
logger.info(f'    → fact_resultado_carrera: {time.time()-t0:.1f}s')

In [75]:
# Diagnóstico completo de carga en Aurora
with engine.connect() as conn:

    # 1. Confirmar base de datos activa
    db = conn.execute(text("SELECT current_database(), current_schema()")).fetchone()
    print(f"Base de datos: {db[0]}  |  Schema activo: {db[1]}")

    # 2. Confirmar que el schema f1_dw existe
    schemas = conn.execute(text("""
        SELECT schema_name FROM information_schema.schemata
        WHERE schema_name = 'f1_dw'
    """)).fetchall()
    print(f"Schema f1_dw existe: {'✅ Sí' if schemas else '❌ No'}")

    # 3. Listar tablas en f1_dw
    tablas = conn.execute(text("""
        SELECT table_name FROM information_schema.tables
        WHERE table_schema = 'f1_dw'
        ORDER BY table_name
    """)).fetchall()
    print(f"Tablas en f1_dw: {[t[0] for t in tablas]}")

    # 4. Contar filas en cada tabla
    for tabla in [t[0] for t in tablas]:
        n = conn.execute(text(f"SELECT COUNT(*) FROM f1_dw.{tabla}")).scalar()
        print(f"  {tabla:<35} {n:>8,} filas")

    # 5. Confirmar el engine apunta a la misma BD que DBeaver
    host = conn.execute(text("SELECT inet_server_addr(), inet_server_port()")).fetchone()
    print(f"\nServidor Aurora: {host[0]}:{host[1]}")

Base de datos: postgres  |  Schema activo: public
Schema f1_dw existe: ✅ Sí
Tablas en f1_dw: ['dim_circuito', 'dim_constructor', 'dim_estado', 'dim_piloto', 'dim_tiempo', 'fact_resultado_carrera']
  dim_circuito                              77 filas
  dim_constructor                          212 filas
  dim_estado                               139 filas
  dim_piloto                               861 filas
  dim_tiempo                             1,125 filas
  fact_resultado_carrera                26,668 filas

Servidor Aurora: 172.31.38.240:5432


## 7. VALIDATE — Validaciones post-carga

*validate()* — 3 tipos de validación post-carga:

Corre directamente en Aurora (no en pandas) para validar que lo que está en la BD es correcto, no solo lo que se intentó cargar.

* Validación 1 — Conteo BD vs CSV: para cada tabla cuenta las filas en Aurora y las compara con las filas del CSV original. Si hay diferencia, algo se perdió o duplicó en el camino.

* Validación 2 — FKs huérfanas: hace un LEFT JOIN entre la fact y cada dimensión. Si el JOIN no encuentra match (WHERE d.pk IS NULL), esa FK está huérfana — apunta a un SK que no existe en la dimensión. No debe haber ninguna.

* Validación 3 — Nulos críticos: verifica que ninguna de las columnas clave de la fact (las 5 SKs y puntos) tenga valores nulos, porque son columnas que deben tener valor en todo registro válido.

In [76]:
with engine.connect() as conn:

    print("=" * 55)
    print("  VERIFICACIÓN POST-CARGA — f1_dw")
    print("=" * 55)

    tablas = [
        "dim_piloto", "dim_constructor", "dim_circuito",
        "dim_tiempo", "dim_estado", "fact_resultado_carrera"
    ]
    print(f"\n{'Tabla':<35} {'Filas':>8}")
    print("-" * 45)
    for tabla in tablas:
        n = conn.execute(text(f"SELECT COUNT(*) FROM f1_dw.{tabla}")).scalar()
        print(f"  {tabla:<33} {n:>8,}")

    # PKs duplicadas
    dupes = conn.execute(text("""
        SELECT COUNT(*) FROM (
            SELECT piloto_sk, tiempo_sk
            FROM f1_dw.fact_resultado_carrera
            GROUP BY piloto_sk, tiempo_sk
            HAVING COUNT(*) > 1
        ) t
    """)).scalar()
    print(f"\n  PKs duplicadas en fact: {'✅ 0' if dupes == 0 else f'❌ {dupes}'}")

    # FKs huérfanas
    fks = [
        ("piloto_sk",      "dim_piloto",      "piloto_sk"),
        ("constructor_sk", "dim_constructor", "constructor_sk"),
        ("circuito_sk",    "dim_circuito",    "circuito_sk"),
        ("tiempo_sk",      "dim_tiempo",      "tiempo_sk"),
        ("estado_sk",      "dim_estado",      "estado_sk"),
    ]
    print(f"\n  {'FK':<20} {'Huérfanas':>10}")
    print("  " + "-" * 32)
    for fk, dim, pk in fks:
        h = conn.execute(text(f"""
            SELECT COUNT(*) FROM f1_dw.fact_resultado_carrera f
            LEFT JOIN f1_dw.{dim} d ON f.{fk} = d.{pk}
            WHERE d.{pk} IS NULL
        """)).scalar()
        print(f"  {fk:<20} {'✅ 0' if h == 0 else f'❌ {h:,}'}")

    print("\n" + "=" * 55)

  VERIFICACIÓN POST-CARGA — f1_dw

Tabla                                  Filas
---------------------------------------------
  dim_piloto                             861
  dim_constructor                        212
  dim_circuito                            77
  dim_tiempo                           1,125
  dim_estado                             139
  fact_resultado_carrera              26,668

  PKs duplicadas en fact: ✅ 0

  FK                    Huérfanas
  --------------------------------
  piloto_sk            ✅ 0
  constructor_sk       ✅ 0
  circuito_sk          ✅ 0
  tiempo_sk            ✅ 0
  estado_sk            ✅ 0



In [77]:
def validate(raw, engine, schema='f1_dw'):
    errores = 0
    with engine.connect() as conn:
        print('\n Conteo BD vs CSV:')
        for tabla, src in [('dim_piloto','drivers'),('dim_constructor','constructors'),
            ('dim_circuito','circuits'),('dim_tiempo','races'),
            ('dim_estado','status'),('fact_resultado_carrera','results')]:
            real = conn.execute(text(f'SELECT COUNT(*) FROM {schema}.{tabla}')).scalar()
            esp  = len(raw.get(src, pd.DataFrame()))
            ok   = 'OK' if real==esp else 'DIFERENCIA'
            print(f'  {ok} {tabla:<35} BD:{real:>8,}  CSV:{esp:>8,}')
            if real!=esp: errores+=1
        print('\n FKs huerfanas en fact:')
        for fk,dim,pk in [('piloto_sk','dim_piloto','piloto_sk'),
            ('constructor_sk','dim_constructor','constructor_sk'),
            ('circuito_sk','dim_circuito','circuito_sk'),
            ('tiempo_sk','dim_tiempo','tiempo_sk'),
            ('estado_sk','dim_estado','estado_sk')]:
            h = conn.execute(text(f'SELECT COUNT(*) FROM {schema}.fact_resultado_carrera f '
                f'LEFT JOIN {schema}.{dim} d ON f.{fk}=d.{pk} WHERE d.{pk} IS NULL')).scalar()
            print(f'  {"OK" if h==0 else str(h)+" huerfanas"} fact.{fk} -> {dim}')
            if h>0: errores+=1
        print('\n Nulos en columnas criticas de fact:')
        for col in ['piloto_sk','constructor_sk','circuito_sk','tiempo_sk','estado_sk','puntos']:
            n = conn.execute(text(f'SELECT COUNT(*) FROM {schema}.fact_resultado_carrera WHERE {col} IS NULL')).scalar()
            print(f'  {"OK" if n==0 else str(n)+" nulos"} {col}')
            if n>0: errores+=1
    print(f'\nResultado: {"TODAS LAS VALIDACIONES PASARON" if errores==0 else str(errores)+" ERRORES"}')
    return errores==0

validate(raw, engine)


 Conteo BD vs CSV:
  OK dim_piloto                          BD:     861  CSV:     861
  OK dim_constructor                     BD:     212  CSV:     212
  OK dim_circuito                        BD:      77  CSV:      77
  OK dim_tiempo                          BD:   1,125  CSV:   1,125
  OK dim_estado                          BD:     139  CSV:     139
  DIFERENCIA fact_resultado_carrera              BD:  26,668  CSV:  26,759

 FKs huerfanas en fact:
  OK fact.piloto_sk -> dim_piloto
  OK fact.constructor_sk -> dim_constructor
  OK fact.circuito_sk -> dim_circuito
  OK fact.tiempo_sk -> dim_tiempo
  OK fact.estado_sk -> dim_estado

 Nulos en columnas criticas de fact:
  OK piloto_sk
  OK constructor_sk
  OK circuito_sk
  OK tiempo_sk
  OK estado_sk
  OK puntos

Resultado: 1 ERRORES


False

## 8. Resumen final de carga

Consulta de conteo final en todas las tablas:

Hace un SELECT COUNT(*) en las 6 tablas de f1_dw y los imprime en una tabla resumen. Es la "foto final" del pipeline: cuántos registros quedaron en cada tabla de Aurora después de todo el proceso. Útil para documentar en el README y para la defensa del proyecto.

In [78]:
with engine.connect() as conn:
    print('Resumen final en f1_dw:\n')
    for t in ['dim_piloto','dim_constructor','dim_circuito','dim_tiempo','dim_estado','fact_resultado_carrera']:
        n = conn.execute(text(f'SELECT COUNT(*) FROM f1_dw.{t}')).scalar()
        print(f'  {t:<35} {n:>10,} filas')

Resumen final en f1_dw:

  dim_piloto                                 861 filas
  dim_constructor                            212 filas
  dim_circuito                                77 filas
  dim_tiempo                               1,125 filas
  dim_estado                                 139 filas
  fact_resultado_carrera                  26,668 filas
